# **<p style="text-align: center;">Cycle GANs</p>**

### ■ __Overview__
In this notebook, we will investigate the training of two different kind of Generative Adversarial Networks (GANs). The first one is a Deep Convolutional Generative Adversarial Networks (DCGAN) for celebrity face generation. The second one is a Cycle Generative Adversarial Networks (CycleGAN) trained for a map translation.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from glob import glob
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision
from torchvision import transforms as T
from data.dataloader import dataloader
from dcgan.generator import Generator
from dcgan.discriminator import Discriminator
from dcgan.methods import weight_init, training
from torchinfo import summary
from PIL import Image
from utils import create_gif, set_seed

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.__version__, device

### ■ **<a name="dataset">2. Dataset</a>** [(&#8593;)](#content)
#### **2.1. About Maps dataset**

In this section, we will prepare our dataset. For this lab, we will use the `Maps` dataset from CycleGAN, which contains paired images of satellite and map views of the same locations. We’ll load the dataset, preprocess it, and create data loaders for training and testing.

#### **2.2. Download Maps dataset**
We start by downloading the `Maps` dataset. Run the code bellow to download the dataset, it will take some time.

In [ ]:
# ## "Available datasets: ["apple2orange", "summer2winter_yosemite", "horse2zebra", "monet2photo", "cezanne2photo", "ukiyoe2photo", "vangogh2photo", "maps", "cityscapes", "facades", "iphone2dslr_flower", "ae_photos"]
# dataset_name = "maps"
# URL = "http://efrosgans.eecs.berkeley.edu/cyclegan/datasets/{}.zip".format(dataset_name)
# ZIP_FILE = "./{}_dataset.zip".format(dataset_name)
# TARGET_DIR= "./{}_dataset/".format(dataset_name)

# !wget -N $URL -O $ZIP_FILE
# !mkdir $TARGET_DIR
# !unzip $ZIP_FILE -d $TARGET_DIR
# !rm $ZIP_FILE

The dataset is already divided into three parts: a training set, a validation set and a test set for both satellite images and maps.

#### **Define datasets & dataloaders**
Now, we define the maps dataset and dataloaders for training the Cycle-GAN.

In [ ]:
from data.dataset import MapsDataset
from torch.utils.data import DataLoader


## define data paths
train_dir_satellite = "/content/drive/MyDrive/Colab Notebooks/IMDA/maps/trainA/"
train_dir_maps = "/content/drive/MyDrive/Colab Notebooks/IMDA/maps/trainB/"

val_dir_satellite = "/content/drive/MyDrive/Colab Notebooks/IMDA/maps/valA/"
val_dir_maps = "/content/drive/MyDrive/Colab Notebooks/IMDA/maps/valB/"

test_dir_satellite = "/content/drive/MyDrive/Colab Notebooks/IMDA/maps/testA/"
test_dir_maps = "/content/drive/MyDrive/Colab Notebooks/IMDA/maps/testB/"

## define transformation
transforms = None
# transforms = T.Compose([
#                         T.Resize((256, 256)), # resize images to w=256 & h=256
#                         T.ToTensor(), # transform numpy array to torch tensor
#                         T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), # normalize images values to (-1, 1) interval
#                     ])


## create training & validation & testing sets
train_dataset = MapsDataset(root_satellite=train_dir_satellite,
                            root_maps=train_dir_maps,
                            transform=transforms)

valid_dataset = MapsDataset(root_satellite=val_dir_satellite,
                            root_maps=val_dir_maps,
                            transform=transforms)

test_dataset = MapsDataset(root_satellite=test_dir_satellite,
                           root_maps=test_dir_maps,
                           transform=transforms)

## create train & validation dataloaders
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)

In [ ]:
## print some stats
print('The number of training samples   :', len(train_dataset))
print('The number of validation samples :', len(valid_dataset))
print('The number of testing samples    :', len(test_dataset))

print('The number of training batches   :', len(train_loader))
print('The number of validation batches :', len(valid_loader))
print('The number of testing batches    :', len(test_loader))

Let's plot some samples of the dataset.

In [ ]:
from cyclegan.utils import display_satellite_map_pairs

## check that the loaders work well by extracting an element from the laoder.
## out is a dictionary that contains a batch data from the loader.
out = next(iter(train_loader))

print('> Number of samples in batch:', len(out['satellite_imgs']))
print('> Shape of samples          :', out['satellite_imgs'][0].shape)
print('> min value:', out['satellite_imgs'].min(), '| max value:', out['satellite_imgs'].max())

display_satellite_map_pairs(out['satellite_imgs'], out['maps_imgs'], suptitle='Example of samples from the Maps dataset.')

### ■ **<a name="model">3. Cycle-GAN Model</a>** [(&#8593;)](#content)
Now, let’s define the CycleGAN model. CycleGANs consist of two generators and two discriminators. The generators learn to translate images from domain `X` to domain `Y` and vice versa, while the discriminators aim to distinguish between real and generated images in both domains. We'll define the architecture for both the generator and the discriminator, using a standard architecture for CycleGAN.

![Alt text](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fengineering.nordeus.com%2Fcontent%2Fimages%2F2020%2F10%2Fcyclegan-diagram.png&f=1&nofb=1&ipt=4f471303260eef1f7d508231ea10b05d0a7effa5fff92aff525315f5daa0f7f2&ipo=images)

In [ ]:
from torch.optim import Adam

## Hyperparameters
lr = 1e-5  ## Learning rate
lambda_cycle = 10  ## Weight for cycle consistency loss
num_epochs = 100   ## Number of training epochs

## Initialize discriminators for both domains (DX for domain X, DY for domain Y)
disc_DX = Discriminator(in_channels=3)
disc_DY = Discriminator(in_channels=3)

## Initialize generators (G maps domain X -> Y, F maps domain Y -> X)
gen_G = Generator(img_channels=3,num_residuals=9)
gen_F = Generator(img_channels=3,num_residuals=9)

## Optimizers for discriminators and generators
opt_disc = Adam(
    list(disc_DX.parameters()) + list(disc_DY.parameters()),
    lr=lr,
    betas= (0.5,0.999)
)
opt_gen = Adam(
    list(gen_G.parameters()) + list(gen_F.parameters()),
    lr=lr,
    betas= (0.5,0.999)
)

## Loss functions
mse = nn.MSELoss()
L1 = nn.L1Loss()

In [ ]:
from cyclegan.methods import training

training_output = training(
    gen_G, gen_F, disc_DX, disc_DY,
    train_loader=train_loader, valid_loader=valid_loader,
    opt_gen=opt_gen, opt_disc=opt_disc,
    criterion_0=mse, criterion_1=L1,
    device=device,
    lambda_cycle=lambda_cycle,
    epochs=num_epochs
)

### ■ **<a name="eval">4. Model Evaluation</a>** [(&#8593;)](#content)

In this section, we evaluate the performance of the CycleGAN model using several key metrics to assess the quality of the generated images. These metrics provide insight into how well the model has learned to map between the satellite and map domains.

In [ ]:
from cyclegan.utils import plot_predictions

best_gen_G = Generator(img_channels=3,num_residuals=9)
best_gen_G.load_state_dict(training_output['best_gen_G'])

best_gen_F = Generator(img_channels=3,num_residuals=9)
best_gen_F.load_state_dict(training_output['best_gen_F'])

plot_predictions(best_gen_G, best_gen_F, test_loader, device)

#### **4.2. Evaluation Metrics**
The evaluation includes the following metrics:

1. **Root Mean Square Error (RMSE)**:  
   
   $RMSE = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2}$
   
   RMSE measures the pixel-wise difference between the ground truth and the generated image. Lower RMSE values indicate better accuracy in reproducing the target image.

2. **Structural Similarity Index Measure (SSIM)**:  
   
   $SSIM(x, y) = \frac{(2 \mu_x \mu_y + c_1)(2 \sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}$
   
   SSIM evaluates the perceptual similarity between two images, taking into account luminance, contrast, and structure. A higher SSIM value indicates more similarity between the ground truth and the generated image.

3. **Peak Signal-to-Noise Ratio (PSNR)**:  
   
   $PSNR = 10 \cdot \log_{10}\left(\frac{MAX_I^2}{MSE}\right)$
   
   PSNR quantifies the quality of an image by comparing the maximum possible pixel value to the mean squared error between the original and generated images. Higher PSNR values indicate better image quality and less distortion.

These metrics collectively offer a comprehensive evaluation of the model's ability to generate realistic and high-quality images across the satellite and map domains.

In [ ]:
from cyclegan.methods import evaluate_models

evaluate_models(best_gen_G, best_gen_F, test_loader, mse, device)